# EDP2 starter notebook

Make sure you choose the **latest Weekly release** when you run this notebook on the RSP. This will give you the latest software versions with bug fixes and performance improvements.

## Setup

In [ ]:
import lsdb

import astropy.units as u
from astropy.coordinates import SkyCoord
from upath import UPath
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Setup
from dask.distributed import Client

client = Client(n_workers=4, memory_limit="4GiB", threads_per_worker=1)
print(f"Dask is running at: {client.dashboard_link}")

## Open catalog

In [ ]:
# Open DP2 catalog and plot
# Path on RSP
base_path = UPath("/rubin/lsdb_data")
cat = lsdb.open_catalog(base_path / "object_collection")
cat.plot_pixels();

In [ ]:
# Choose catalog
# Object catalog: "object_collection"
# DiaObject catalog: "dia_object_collection"
cat = (
    lsdb.open_catalog(
        base_path / "object_collection",
        # Select columns
        # Column descriptions available at https://sdm-schemas.lsst.io/dp2.html
        columns=["g_psfMag", "r_psfMag"],
        # Select a sky region
    )
    .cone_search(
        ra=10.0,
        dec=-5.0,
        radius_arcsec=2 * 3600,
        # Query on column values
    )
    .query("g_psfMag < 28.0 and r_psfMag < 28.0")
)

In [ ]:
# Check catalog
cat

## Plot skymaps

In [ ]:
# Plot catalog
cat.plot_pixels();

In [ ]:
# Plot zoomed catalog
import astropy.units as u

fov = (8 * u.deg, 8 * u.deg)
center = SkyCoord(10 * u.deg, -5 * u.deg)
fig, ax = cat.plot_pixels(projection="AIT", fov=fov, center=center);

In [ ]:
# Plot density
import hats

hats.inspection.plot_density(cat.hc_structure, ec="face", projection="AIT", fov=fov, center=center);

## Map a function

In [ ]:
# Map a function


# The mappable function takes in a dataframe that represents a partition.
# It returns a dataframe.
def g_minus_r_mapper(df):
    df["g_minus_r_mag"] = df["g_psfMag"] - df["r_psfMag"]
    return df


cat = cat.map_partitions(g_minus_r_mapper)

## Compute catalog

In [ ]:
# Check catalog before computing
cat.head()

In [ ]:
# Compute catalog and write to disk
cat.write_catalog("dp2_example_cat", overwrite=True)

In [ ]:
# Compute catalog to dataframe
df = cat.compute()

In [ ]:
df

In [ ]:
# Read catalog from disk
cat = lsdb.open_catalog("dp2_example_cat")

## Crossmatch

In [ ]:
# Crossmatch
des_cat = lsdb.open_catalog("s3://stpubdata/mast/public/des/hats/des_dr2/", columns=["MAG_AUTO_G"])
des_cat

In [ ]:
des_cat.plot_pixels();

In [ ]:
# Crossmatch LSST catalog with Gaia catalog
x_cat = cat.crossmatch(
    des_cat,
    suffix_method="overlapping_columns",
    # Filter out extraneous matches: match rows should have similar magnitude
).query("-1.0 < g_psfMag - MAG_AUTO_G < 1.0")

In [ ]:
x_cat.head()

In [ ]:
x_cat.plot_pixels(projection="AIT", fov=fov, center=center);

In [ ]:
x_df = x_cat.compute()

In [ ]:
x_df

In [ ]:
plt.figure(figsize=(14, 10))
plt.hist2d(x_df["g_psfMag"], x_df["MAG_AUTO_G"], bins=800, cmap="plasma")
plt.colorbar(label="density")
plt.xlabel("$g$ magnitude (LSST)")
plt.ylabel("$g$ magnitude (DES)")
plt.title("$g$ magnitudes, LSST vs DES");

In [ ]:
## Close Dask client

In [ ]:
# Clean up
client.close()